In [1]:
%pip install -q transformers accelerate huggingface_hub langdetect

Note: you may need to restart the kernel to use updated packages.


In [2]:
from huggingface_hub import login
import gc
import re
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, GenerationConfig, BitsAndBytesConfig
from langdetect import detect

print(f'PyTorch Version: {torch.__version__}')
print(f'CUDA Available: {torch.cuda.is_available()}')
gc.collect()

api_token = input("Enter your Access Token: ")
login(api_token)

PyTorch Version: 2.12.0+cpu
CUDA Available: False


In [3]:
class NotebookTranslator:
    def __init__(self):
        print("Loading Gemma Engine...")
        self.model_name = "Qwen/Qwen2.5-1.5B-Instruct"
        self.tokenizer = AutoTokenizer.from_pretrained(self.model_name)
        
        if torch.cuda.is_available():
            print("Loading model in hardware-optimized 4-bit quantization mode...")
            quant_config = BitsAndBytesConfig(
                load_in_4bit=True,
                bnb_4bit_compute_dtype=torch.bfloat16,
                bnb_4bit_quant_type="nf4",
                bnb_4bit_use_double_quant=True
            )
            self.model = AutoModelForCausalLM.from_pretrained(
                self.model_name,
                quantization_config=quant_config,
                device_map={"": 0} 
            )
        else:
            self.device = torch.device("cpu")
            self.model = AutoModelForCausalLM.from_pretrained(
                self.model_name,
                dtype=torch.bfloat16, 
                device_map="cpu",
                low_cpu_mem_usage=True
            )
            print("Hardware Target Locked: CPU (Optimized Low-RAM Mode)")
            
        if self.tokenizer.pad_token_id is None:
            self.tokenizer.pad_token_id = self.tokenizer.eos_token_id

        self.LANG_MAP = {
            "en": "English", 
            "fr": "French", 
            "es": "Spanish", 
            "de": "German", 
            "da": "Danish", 
            "zh": "Chinese"
        }
                
    def _flush_memory(self):
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()

    def detect_language(self, text):
        try:
            return detect(text) if (text and any(c.isalpha() for c in text)) else "en"
        except Exception:
            return "en"
            
    def translate(self, text, source, target):
        self._flush_memory()
        tgt_lang = self.LANG_MAP.get(target, target)
        
        prompt_transl = (
            f"Translate the following text strictly into {tgt_lang}. "
            f"Provide only the direct translation output with no introductory text, notes, or explanations.\n\n"
            f"Text to translate:\n\"{text}\""
        )
        
        messages = [{"role": "user", "content": prompt_transl}]
        formatted_prompt = self.tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
        inputs = self.tokenizer(formatted_prompt, return_tensors="pt").to(self.model.device)
        
        with torch.no_grad():
            generated_tokens = self.model.generate(
                **inputs,
                max_new_tokens=350,
                do_sample=False,
                pad_token_id=self.tokenizer.pad_token_id
            )
            
        input_length = inputs.input_ids.shape[1]
        raw_output_text = self.tokenizer.decode(generated_tokens[0][input_length:], skip_special_tokens=True)
        return raw_output_text.strip().strip('"')

    def generate_ai_response(self, prompt_text, max_new_tokens=550):
        self._flush_memory()
        messages = [
            {"role": "system", "content": "You are a helpful, direct assistant. Answer the user's questions accurately and concisely without any fluff."},
            {"role": "user", "content": prompt_text}
        ]
        
        formatted_prompt = self.tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
        inputs = self.tokenizer(formatted_prompt, return_tensors="pt").to(self.model.device)
        
        with torch.no_grad():
            generated_tokens = self.model.generate(
                **inputs,
                max_new_tokens=max_new_tokens,
                temperature=0.7,
                top_p=0.95,
                do_sample=True,
                pad_token_id=self.tokenizer.pad_token_id
            )
            
        input_length = inputs.input_ids.shape[1]
        response_text = self.tokenizer.decode(generated_tokens[0][input_length:], skip_special_tokens=True)
        return response_text.strip()
    
# Instantiate the translator module
translator = NotebookTranslator()
print("Multilingual Engine ready and optimized for speed!")

Loading Gemma Engine...


Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

Hardware Target Locked: CPU (Optimized Low-RAM Mode)
Multilingual Engine ready and optimized for speed!


In [4]:
def extract_questions(text):
    return re.findall(r'\d+\..*?\?', text) or [text]

In [22]:
print("\n" + "="*40)
enquiry_text = input("Enter your enquiry: ").strip()  
# Added target language preference input
target_pref = input("Enter desired target language code (e.g., 'en', 'fr', 'es', 'de'): ").strip().lower()
print("="*40)

In [14]:
# DETECT SOURCE LANGUAGE 
detect_lang = translator.detect_language(enquiry_text)
translated_enq = translator.translate(enquiry_text, detect_lang, target_pref)

# Print results
print(f"Original Enquiry: {enquiry_text}")
print(f"Detected Language: {detect_lang.upper()}")
print(f"Translated Response Target ({target_pref.upper()}): {translated_enq}")

Original Enquiry: List 5 random celebrities and if they are relatively perceived as underrated or popular by the general public
Detected Language: EN
Translated Response Target (DE): Ziehe 5 zufällige Star aus der Populärkultur und prüfe, ob sie in der Allgemeinheit als übersehen oder populär wahrgenommen werden.


In [23]:
# QUESTION SEARCH
def contains_question(text):
    return '?' in text.strip()

print(f"Contains Question? {contains_question(enquiry_text)}")

Contains Question? False


In [25]:
# Response in chosen tongue
ai_response = translator.generate_ai_response(enquiry_text) 
print(f"AI Response:\n{ai_response}\n")

# Re-translated response in user's preferred language
response_transl = translator.translate(ai_response, detect_lang, target_pref)
print(f"Translated Response ({target_pref.upper()}):\n{response_transl}")

AI Response:
1. **Sofia Richie** - Overrated: Sofia Richie is often criticized for her lack of substance in her acting roles, and some feel she lacks depth compared to more established actresses like Jennifer Aniston.
2. **Taryn Manning** - Underrated: Taryn Manning has been overlooked despite her talent and impressive performances, particularly noted for her role in "The Social Network."
3. **Natalie Portman** - Underrated: Although Natalie Portman has received critical acclaim for her work, including Oscar wins, many fans perceive her as an underappreciated actress due to her diverse career that spans film, theater, television, and music.
4. **Dwayne Johnson (The Rock)** - Underrated: Dwayne Johnson’s versatility on screen is widely recognized, but he is sometimes seen as a one-trick pony in certain genres compared to his contemporaries who have broader appeal across different types of films.
5. **Keanu Reeves** - Overrated: Despite being considered a legend by some, Keanu Reeves’ pe